# Sparse Walker systems — A100 long-context benchmark

This is the normal multi-cell version.

**Experiment A — cached streaming:** SASRec, HSTU-core/large, Sparse Walker state,
and Sparse Walker + degree-128 terminal at 1k/2k/5k/10k/20k/50k/100k.

**Experiment B — paper-shape sanity:** BF16, d=512, 8 heads, 1k/2k/4k/8k,
HSTU-style attention core vs Flash/SDPA Transformer attention.

Triton kernels are written to a real `.py` module in the setup section because
Triton 3.6 requires inspectable source files.

In [ ]:
import torch, pandas as pd
from IPython.display import display

print("torch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))

## 1. Load model + kernel definitions

In [ ]:
import base64, gzip, importlib.util, sys
from pathlib import Path

payload = 'H4sIALOdeGoC/+09a3PbSI7f9St4ntotyqEVSXls4htdrRMpsSuO44mVzE55XSxabFlcUaRMUrQ9W3u//YDuZr/Y1MPJ7KNuUkkkkWg0GkCjAfQrWizTrHAWQTHznCJakNY0SxdOGBTBJA7ynOROxEDEo1aLP0lWi+WDE+ROsqweLYMkhAfwdxlWz4o0m8y0H50kocUS82lnukomRZQmQYwA71qt4ejryduRM+AgISmjCXH3Jqsw2HOiKX+MPztR7gdlEMXBdUzctkPinDh7k+Vqr93ChkA1DFmneFgSZzBwGBbP2fsCkIHz/vyL8zaF4k62SpAVnb1Wi+G/DiZzkoQ5qwiYtVjFnSCO0zu/mD7rA3njbEUswNAkC1j2cNhy4A+Dz0nhT+M0KJ71fYbaX2ZkEuXAB3dvFt3MoAXkfkKWhTOiH/CCIVhScSyzKCncPYrtENrD0Pp+STLE4fvtCuTtl+GRhODvaaMECHBBQtDm3gB9jO9+EiyI2223W7INlQSzqEiT+pNOHCQ3q+CGoDyLmAIcH134488n409nFUdoW1j9Y1qMksAQaO0w2YBYyWEd67sApF9H66wSoSJYRUaWmUuwPW+Pzo/enox/gbK9btfvdnut45OL8afPJ6MLeOb24FHXc/rs4wX76HX5U/75ols9p1/arTdH47fHiLL189Hnj1/O4euzbutkPPqMSPvdbms4/uVcqjdXg1aLdjTn3bszFzToYxquYtJmzQzJ1PH9KIkK33dzEk89J/ScKElIxiHwT75aksxtdwRkW76CMp2EFFAp4L4gtysC2h7EroDAP/DqNEpIkLkCu2cCvB+dfnHrj3k5WgiIUwDaogXTNLsLspA34F6hPCPFKksEle49CIez4+Lo4k2cTuZb8WTw8rnnzEgQ5oM+b8Gg/+LlLjzqMRadBg8kO0uzhRsaEEFRJAzm4youIqztqCiQndBzNbaEnBYgLEuX6aoYdDugKNdBMZn50yjLi4HoCJJTkpT+JlKmU6QEFUbIS+XbBcnKKLnZhnOJHxVkkQ96fb/X7XnOJFgGk6h4GFRdZBcWMlxIPPumv65Qw/vqqw4Q+os0JDG8f/lcf4PYGE9Gi2sShrRxvLYnDpCN4l8G9IUfhfeDrkHaMs3N8hUNWNiAvka94wUYA0+jvHAvhUq2UaUdH1jvZGDxiNtvX5nMALmZYsSKWq0/y7FVSOxtMJlxyzY/dGKojX4vle8xSW6K2SHUWQASFOUimBM/D3J/gqVdyjzQPCiQZg9cbnPPKYGMyysP/tEnSDltIFJPy/D2SkGjEaPPqM53YOT3qT4LgOEMIFjZSmhPnzrH4j0UXC5hVHSZoQMehUYXobbSc+AvQyOlMZzpNoYNRgM2mkOPwvF8QA2pYmqcfafb6UsRlP9KArhRq8TqogykXLjwlln6NzIp/Nt56S5mgef8yiU2BNb+2slnwZJcHvSYzH4G44Ecn4Hfk/hY1L8j4CoUnvbsOgqYjG7ROnRiZpp/9ZyfLw+HoAHX+MFonNcghof9fQbEvjG4sgYH7w4pGP3CoI45cbqmUC0ZSs3gfJFyuO2UEblzobXRr+hpUHEMZ8oYMt8IUa6FoP2N6UCUTElGEvBrUOBgu1AK2HvygiyrzkN7EphUsCxgR7hEmPGgrzqsF9LH96IPILhblQGDxJ5CsQ6T0iV8veqskhxGX0KppAhOAQHiBgvWEl0zjjzZPQkwlGRBwTs376iKSf5V9NSkh4On4KxX9XxVz2SfRnUTwKxl88s4uro89Bz4C1TB51Vnki4ffHdugpaNoGW7JWADqjr5JIhJ6IdpgToaroCUwD5q3ioc6huORp3Ew1Os1gZVboDiY7K/pKOy9ipCW7rKg3hAXUqlf3ckZZ2M0N7pchOi2UGl/agf9yBcxZLSakEgbtBuAIORnYsp6YNAOTrec1hNOLJQR4mNHscX4y98wN/eX/KrwYK7KbegLWEph/8dRn05avNv+mtaA7zUBxBW8hatEPxvPEa1DcsdnQeyxFp65OCllECRFgES1gfrDFj3GRHAa/oAat+3kbUqKV3AxvMggxCogOYzE0IWy+LBFcyj6NuSHVACWUQFFMSM1xQb6AgJEuYE5kWIX/qKovzg0HEihOqpGJ00iR+chBCgtZgRGPmDLIHXJfTjNANxJWnhBIlzdn/moMnHeD6L7jt6M4rcv1vXjF7/dbvuJa0vg3wTkjhwepsajzTYG7+2GKVjA9MoYMo9LB67CBELDZdusdJNtukhZZBFQVJIB/nZ6xf9b3OPOUogmX97tHv8orvJ8Rbvo6kj6sUcSJqRvUPN5sXonuamKcBe46Fn/YJGuqIEiQ2McZDdbI3ylYdY+y/wn4IyJ3r5LIhy4nwN4hUZZVmaubxCU/wbwwIk/tFhwYvulmGBRnvNHCMNjWZWK2rEFIyLEuTKGjxgfVtED5jhyotgscwPefZhTJI8zdaFFrO8WH332GLr0ICNgpxzpo/OXiI//4mxwjYUld9AUIsNBj1nlq6Aj6S4IwR8wBKcpbxjSBGzVAJDz3/JclD4Tzx9wgUdMHUyCbaSxkrEaXKDVD17ydG1dedV9UmE/vEwR5JYC3moPgnnz6d1MWfHow6rz73vsvqi4mI6CvaY3Bd+UfnA6EIrLjnrz6c4NDEiP8BPjlfzCOnLr2iKbS/p2xxtpcjVkSjJVwt373oWetezJDz4H/h/j9L9gTMkEcCc46cGk9uXZ2lCRPUZidEooT0BglkgwH0/bYytWkbDO+Hv0zHykuPgJIOAC/S6OY+wUVjjFaCRrKRNPWUkXK8mc1LoylSpwA3vD8F17lLE7Q5YnAWESlEy6LVZ9tJtt52noL3Puj2uKKg7bgWK2dH+q7bQXb0J6B1cchJ4C+5YzBDFK5fxH1gimv5E4lAVskdp0BmnaqgpwEQIMAQBgp/xdVOUSBUXw0Sus48IFtVYj4Yqq8wvJKQuHgwXmUoxQSIYg3/C+uTa+HOfzu9AwJIVrh6bbBeafkMoykWnmcB3HTqQ+SxsgaFCownGDvDcB0x68E0fEf/MxYpOtC1buowj9Kf0EbiyxMAI1WI/FoaGCY8HuhLfVmjZEgIKd8s+5vghjAZtDKZY6BcwHdFicNBTXJ4VgK5YtsM2IFVNUgY2jp9+7lLwlhe83argrZIfqJo037rk9lkIRLlDJoKBa9kI6wBUF6cQkEKSp1ZoyzqoY9WpGKY8i9ayvIjSKTRsgZleUBgndAyEpdNwaYcyKNV6mo20e2GbU3dlEgLKCKgDtCL3mgvwjkdvmIC7r8YFkAdOxXAt7hQpvKI+BsgEQgg5f8csyJ/5/N/fokKGZExekyIqiSK2OYFoWJnA+gmGYDDisj0fRr/44wsY+0Z/GeMX8eL804X/s+eM8X/x8NOXsfxx9HZ88nXkn8knORAWEv8W3OMC3Kc0gXHgfpnVAcINAPNNGObJJoBNVZSbqig3VVFuqqLIN6FINxGRNtZx3PSiirb9JoDhT41vvja9eXP66e0H/2z962bE/L0VvTIszmBMRhsPg26W3mQBjtGuElly9U4YDPSd0K2UkPYaeAhh2TMlXmEehignfF3abW4ZHu6Bgv9VNUOx7kpVmnn4CXo2JXdfaj08ApzyQahblEWQzwcA8CNIwHPSYkYyPatatYHPdSsGmSYEbKR+VZkzYUC/kizN3csK4krEKwpmkVISTlPVSm4Immn5wfnMFoE4cZouD527NJs7N1l6lzt3UTGTZoGm3oQ2yowbuktg/bMCXSWoQrSpElPVurO2nupAubOCT2zMONP9oRKMbEgDhh8F5pYGMZ83SZdaxtoTU+LzmQUkkZGEBEwsgOGtCHEUSENnhN7Q1kjUf3RctTyqVLte0qJj6/VMi+XiDgYBc/TWZFUgpfsoH/SMQhnNtSmxmZC7DMtEn7xex3Y67gAGwLiOFd+psXPyoKt/XQ3oAImClVICw/4o2nahrIpPAWABLF+AKFDnIcCs+uwBpx7ch16nq/OYRYoYq9KG3fCotIo/G6ylVhIrjpKq4ooG9tpzcPJQhKsiHFYlO2ZyrAoo7JGiWs8CjJKqyBaUBjr9tfYenWf3VxALqmp0s0hhtPiVBrhC+2rwAHoHlROXkwKeK9KhV1yW6zTi60bDUG5rGEqrYSgthqHcwTCUmmH4+r0MAw4wT4RduFObU5bcLqicREAIG4jOQXAha/xK6dBZKg+M1kLVtrG0pO1TnXN78oy6y9+YPZPrPOQ0WLoqZNKLTvrIlRIix7kueaiNx6p3Q98WNOvsXjKCrtZi4j2ZZaqoZ8NyJ3ezKCb0yY8ykpTDKr7YHzh9Xq6slStluVItVrJi9Mna2OPSPfbaV8o6hsvulZAA/V7K71J3ZI4HQWoRopLX85QMmQQA0WgIhW+hEdJh+kaXQqi/e+qyCkGeCl1/2rM+VSfnSyum0oqpXItJY4+CzGi+Wo3+RG3g8eDYEj0M9CShGj4MLDMKw6+Delaf+2W44LAWKgxA9eoBAjxVyuMSmbsgg0j8ebVERQmloUVor25FqLxtXpKZg98Tk78nJn9PTP7nJyZZb/49M/lPz0yiVQWeTeZs/MfJLH+SJgXybBmAWB5cblksOUw5e7YIklUQ+zkhodvrP1OUg+Ku1nGoa1LY0gxKDp8t7BDwg12zLJWpv0TrYk7SK8j53gV7YVAvXjmbuNXldd+ZQLMJX+N8jyZWr7kzN4S3RRFT3sZrqaIVprXg3Ju1qAlfEaL7mr0/mX6m0v0eGC/NKT6Nl3r1bGBtKxgYQ2ujcSMOeG8iUXbg5A/JZJalCSqtQihEqz6EylAR02OXUn7Aqm/TKLqNIS1Ovi5XrrI0a5LmUUJEwXcd9sDPo0UUM6VGXLJ/UJRigpeN4go+tsnm73ucor3DijYPFxghanjEvvxDMS5sV1bVjB9xrd4zJ0jCir7/gaj1Nf4xKto7/XT2/uDtp7Px6C9jh2/6OT+Cz18O4fPiYk9Wgv1qregMMdj4T6Me3qU054zu2Wi1fnDGM5ITh8UCuRNAMA+mPSKhQ3fT4ZI9AoFjgUv53gV58XMQA+xnAvWRkmRimd8SvKFDQDeku1cy8P3g5fGg99JzPgxeAa1pQvc/5eDd4m4Wz7nJguUMGnmTEQL+o1Pcpc4shV4DWH4mzgqowuUrSATV/dy5JriglQBRUQ4V0+12uLAuX4ENFZSABAq66rDTav18dPph9Nkfsm0Y/Bfd1vSy+nVxMsRdTEBS9QTXWbyqfrz/fHR+7A9H7z+PEEwgOfNBiG9H52PcDqWi2ld/VdDj0eePJ2dHpxJRry+qYL71h9HoXCPszD8Zjz5eUFjcz7L1VNMdFZFPZVCbYEKc/snQY19GH9+MhsOTs/ee89OX0edf/J9HJ++PxzC2jt6NgaRfLjznMz5h3wWWd59HF8eABl6zrx9BdQGHzzVbiTuMKYXavIiDfGqalzg5O/8y9i/eHp2ONkxNhGYCXt0IMzNf9l4q/q35Evda1YwwTzZx9hn5OW1kt+aldG7D2E7x7jtDTKzw5FuI2RKZfGucdoBiCl9k5Xdx42SIIltM7qipIUqBTB/XsziuAv6jc9ymKSwtg9XegmhJZbYllcecuH8ttZMtqcXVxP9SimVcEcvJAVCJfefeMjkggDMFONsEPFGAJw3AKh1PGTjGyLzYLRJ0G4tsJIbq5OCVRpClFFJ2m60pNbGVQhJvJ82lRAJU2C0UnkfLUNHMqEBkk+LGCSlhLwFDror/mOrDOvHnqvjRGFINmKkacLyTzmaNVEpT/i8kU3Izl+oU01mseI3yZQp0RqGzddoXR93Krt+gIxnnMgMuYMqumMCxA+R9dUoix0XigBiDNNKHj1jJpMRRz6iwb6uxp9bYt2TlM4PwzEJXphJuBbAQnimEq0uxM4PwzEZ4phKeWQmfmuPoc5U5oUrPFJQFUFBWAt9U/tjASgQrVV3Q0QHgHxwYF6GR2LaI/qeCl2ugS/qfpjrpTUTTgUDNE3gnXXiGBvwPl8McCFnS34IrSnxRGSZQ26VpiixzMcKzApCp57gxnYehnuUTbLfuebSbEKA/xjAsMfG6wVXEj5qneDE+Go+Yj8e+Mh+vyfXD4VBx+0YXY3B4xyefzjxnNHw/8ugjtD0eTjExBPiFFpfLnUzvkDnMm5xI0xucm7r4QXJqka/zBjGOMd+rxtXqSkpvMVTnqgUHQRLzhindBe57MItw6c0bTfy6vjaNFHyaOjWQMF3UClTq00iAuu6Gip6NnHLUlCNB8wBQ23euLydJI32WewH960N9uUIS1WbhZQkA/wACbOOEuIGcm5CbAOl0I8yYplHd8lHghQaLMmsGTgzMU4BLmmB1xNOFDqoPw7rRW2Da4QOSAfGD8RSE8RwxhdhsvcbFQkVSn7LmWDt/egFmJ13Up6atVXX6CJ4s+ABjZLyMhkxmaU4SsZRsuopj1/3goYxoSyyLHngR2VnYGipWyqafYqEMwi8WrdrepL9Jvftg6B3tyPfamMhQ2WVY7SznI8Em0EnV7pzpqmAmjEfRvYdiZqs3GhHo7GPl51j8bx5i9yREY2HJSL146akQ9eKCpRbCuWvBQOwip9XK8VB52jwy0mozjDyoaX76lA8Jeso9r97/wfY6N7qkYJCHqO2svkGzktPRlydtnkA1xhKgvFDsoDLmAexNtGYRT14ERaSGljhEaoUal0OtW/lSDbEYcgJt2zr4wsnXvXmLM7827qCCLu6VpWlA7RZr05QFbdVcCOfPE8QHTtCHyg8wjFm2MPpdPWxgqdf0WrpurLoDKCz4YwFXvDb4KfBKltr0dLnQm0Ex7avaL8pDq571vXZt6Mi3N4rJI83h8t/JHIZWc8i8MGZXUJPXW0TJNt2ahdjB8aWlyBoLmGxr+1Qq11k/Rh4SYsbV9Dmzik6Vtkhs9tC2bIu70egoeo7WSg2kciapd5ZvEQ1kJAjxcIfG1HFDZMAT4v7Xo9MvI/j9cXRxcfR+VOWUBZ6zT58/ikQz/fHm5OiinpE+PhkOR2frYgPzwfbp4nUBwtpUshhKtvbw88X2/n1LWRzdZOh1NmMiR11fuEXCUU8u7pJZWuQ3srMiifvQOsWk1fzWu6ZG6LpBKd4pZxqq2ajht2Z58RgTJbWJi3vym3XD1lazAgK4/GfMCigZJGqllL4AGCNMZWAzpSxJkCgZNcVGgy0ayvlWgEGEB7SAop9K4hjzrBO9uFwipqqA0vObWtjrNEopuTZRod1oZlUjIrYsdMIWJmc0ZYztoZb2BU6vANFPoLq6PWUmiVUJWNR6tzCsBckgMg1iny7+qNlXbu808/rl/PzT5zGdCawZSGNxHp0trPZaXLz9BIG8mo+xpVGGTfNu40/jo1P/7dHZ8GQIxFyYBXGycoOFvV6/VSgOElJLpajr1ddaYZ6luabuORQDiSDClrm3hML9WGtOa6vAwhJUbDEM0IDCspi+yTjolkQzD6r0mZ3XgxFrPTUDYc5RzpTqVH3eqQuRnUzaTrbdsmD+G2bDjN0yBE/VWTPLoIAr2xCQCCqo/xpQR5THGizXoeezVEcaOonhS9v8aLtfW9ojGwNqElmcZ9a1mFuKVDc7z8KuKRYEt4RQRaMLEp6wpMLaYszWWMqV7aY4jxHKdB8scaT7z7uY0gXJbuqrHJotIuacz9Vdsm8/fTkbm/aNFtlg4GwWjJZr18wQBf2RVdXU83URYIldzIjC23qfNMTUhJuKYLOzZO8i23aKXvfRXeJ7KTvTALtq76Sj/MTThwTYV0QTtjSKr4FsPIxLaC5f+WRfIaQwia81kyt/jJO0KBogGZeXGcjb5qlcCYTEQUEHNrHYaV/FY9ZqlvsDHW1x6qzhOCmzAAyruNRJ2Z367vxZH4ll64ODOH7ABXYJLt8KcA8IXUZH17AegGLd4GFxdN9qXkBgCq27UXBdHF18JpOnuPyTblbFhWFJWpDrNJ13rOdrNZ6LZKy6okduVcvHttkjZByB1O3r1d+uSPbADzpdR8YzUevxd60/JlOM6B9yuixariTexA+ceVRJ2oUQuS4Zlx1HN6t0lZsHyWXIkH8TwsyOhaseA7rIUBUY9jONoK7n1BYF6n6NW3/vWNYXtrc4b0vZTaZuTG9mMAlvyFZ6v5a0rSnjvPY2qCMy1+fZ7N0pO/7e5PDVqX6Jx/Q9jqbh96ZpQfI8AN9ms82QZuK3ogW7pUkI+NB5Ex27dEZLTfxkLVaA5ba/R0V6TflqSW97WNu3e55tYNjYuZUR9Xt1auPQfJxi8HneW9nf+tyG0twvYMFUpcObUVWashYZHeUMPIqJ3hUd3d5Tb2TluuzcVorP0lQ7wq1InEVhSBI7vuF2+CyelDgDVcWqO6qK1yWWnMuV5JtZ09QBWf1ysuO7EGBvehMJ6apY1nZu97o7i5vFhmxz5aC6lIXqwDK9gyAynfr9nRrVaEv4ttD6bp2rnYnGqU+iKf32TocjNdk0OxuI8JpkwcjR+gydm7SaQDr/2Ot0nacivGgmw64Tnsnjpj3DVXjFCKTn3iPBSuBk2Qdx6fa0ze6mAD35y6v77V69nHCrPdOdtQALo+2ZpletTC/I9tOwjTTo8rK9M0pyf/Cq49U3YuuSrC/za2SEUD/PlP93apDpW3vSTfV039BSrhoQPMOe66B0wxHzWweCeduxSJ/73FJdtqOq7mh6NkfP1mrpfXmGl+TZafPUwcnGHKpVO2qRer8Rw7xdDxVJCVsnVZSMvq5JxDppcukqltouIt52q4S466f29oZhgA3EXn1oNHopUzbF7eM81qDMaYiBkSwxBAUDzqA+rupLQLZSamuytFG3N7RbHaI9Y2L6y9l4sMkvqGcmB+ZgvVMbVa1kVLXEvr8vF5goYFPub8TBwErmy32O6xfbmEqioyJdqvHEmS7h64LeIdjC+cxet/98f7/Pt1SvwsC/BkWfuVPQLyBtsVoO2C1pNAeZ5QN6TxrXcmOVK4NXesA04Uq/ftssWIycHlDeqna3K/AjPFfbJQneDucXEWbJ6K1cDPH1LsAGubQ9ajaykxFQhFDpqKIBtK76+2ujOUp2PK+OKw86JA6WOQmRIOJeV5eF0BOll50gD7IseMBUc65tY/27wLaH89P+Anfwss2+9x184rYVT2hv+aKrggDmJckmmImMCR6b8aKrg79+sRb89QsD/PV68NcV+D/4QUjiwit/cW0/l/76AQyD71QJ13/GhVe8yicDsamNk4SzdzP477kqAQ6tdBJ5nMuWTZPoX2FPlBvot22ypFg/oEEiduURIk/k6Q6b24JNyVaJL5LPrOcvgmxeHZ5QPyXhOZ8tyWkWQ7nDrulEBD7psP0RCvSiCrMEu72iqQgbC5Ae63yF2zAPoZ8Nwa4mxelT1j2pH5QryZp1xxSIAqgb2x9sQE+upIZPqEMlV1AIcdXlobnT/q8Je/fLnnp4vzl3wM4VCR1o0gG9ncBZLXEw7mgrYIzb6eCbglNMMNEThMTgEAeL6zA4lNdy0VITTzBNWfwJbYTCf9fGvT2q62BN9hile/qwuMcva6UAw/Mj8zUnD17zb8Z71jc/vtk71K2Q1jajzE2cXoOSC3+Clq6dUre/rzgG/1DbKMw9fDePYNAe0aMPNp9noEiTTgHRfrS9QNXjRuisY02k9GpKI9at5Fo/IoPiqASMr43lvfWutBEzO79jHeJG/Zsm2+oX8u6A2ptmFWO30B6wY9oOlhmZRvd7NuOwd/4wRqq/QR/1sUMXzX+YRjJD/SiVpEW/VSc5kt9AKddj/l5ayca0f0e1NMTzn6KXDUc98SFxGWTAK+YYHDrkHvgKo8OCOOkSj6XDqdCM3ARZGJM8d9KpGInj4Br8Mn29hS545n8oob6xlcOiC8YoSIljtB1QNHv1bSwW3ZiuIKywgW4QvSl+W0xpKbGd5GvS1zVgrRYYmrAF07UMzDfxvcJ0EPb6r/5z+F/fKmWdzJUJC/PAHggUlLDARNT+bYUb4nbfZdgZBkXwDq8yxHc8Dl5GZYpCD6cd+tWNkpDcDwRz8eSleLVI8gGXKF4HGK8I/ObBML0DlBQ+LegqWC+lY+DnSwhsVku/zH3ujV4hTQxMPHlaK7h3VcNHDeejELLB4Ko6ss5mETAkoEU6vNlKWMAwqeC7UmGp8Gpb/ApLZBXqw8fUonXHb2qN3rF3ru+RrbPVuqDrtB6hz8JW2DSaR/ghnoiAePFGzkWr1To/Oofufjo6ez8+xrO2XOzjntPvPn/lOc+7r196zqveawjoGeDx6Gh4Qc8HY7+Hx+xoMfaLJQHxNLEuf0ITgeyWS14A78sTwe81zRT1Xm59thf1QXACLie3W1wjo18IA4Zu/JdH3AaT/H5dzHaXwSSPvi1muP5mlo+PuPelrZq+0F+s240xq71UdzstaJdGFPsVPfYbPj6u2b4xNJZFU4IWzo9MK1tb3uhiXOeysJzZf5toOww23PfCaZGb2PSDHyyXwLSabnThS7g/enSPQv1Ol/oFK35iXrHCuuim+1X8ZPsbVuh59Umdz8ouyIaLNXR+z2eNl6c08Hv9XSl+0sDzBr7rUaWyRaLhEgid+nLWeMNDA/Xlb0C9fkU8534Ig9wt3bZdgDhzd24E6OwSdxSisg1moKh+Tanprcy6Wjt/rLRBIvkjR12/1EOe7gDYPAdv+u2aN5g0X+7BFM2ENXDyez34JoNqHGy80QKZdEd3lGx3fYVxd4XNTqRNkt/ucotmw6HdclGN1Twfg5dbYDMO+f24t0/nT8tD5/IYsIDdmDFh8l94OHiHniRgv83Cj6M5cW+ZXN58ZKeH0u9nzCuh51iww7Jc7kxMwqh0Afmbj2CjjlnRtU7FJSJQ74dgLWD7H6VrodzboF3aoN6ZcGu7KmGu3dqgXdZgvaNBu5pBu5HBBo53EOiXLKhFlJ9qoeFgWBuAB28+1u9NeHO2w10ITCmWIM3Mt88hq05kNZOsuJG/zyf//5lP/oecemQaQ81AffLxB+diEcSxk5NFkOCRKexgd3Y0tBPcBBH4hCDdMAI2Fg5PgkJr+Vqdzm7nv/9Jcvusy08xFv2+a6xRVyImsGfgpFQhU8OybSVAUu46MLBym6cMRmUjhGwIO1TRapCBrjn8K7viFmh9p5vEvgiKxSqGEvJw8bn4zgbvZZoT96DnOQfq2XxsmJWIoOrYVRbRI2/w35q14ddpGisY77rG3dJ02/1ZVwe463ZwqIJOMI3i2P1fTocxmIMu1Np4RzkiTk2XnCRZ1pWntlO+HlAMmw5t7649sl3PYQJWqJoORK5+W7tnXGo33QzYdOT73jn2qwPar9hcDRvveA863PMc9VR4bLhxJHyXHwjPOqgxTd3Qpdih6pTVSdIRY60TLehehDxcBnzc9ZyL4fkblk3VQ4cztKZa3uLQkj7c+2tCYQ4ujuEDCg2gRWemD7puWwn+0fvwLl14jeM+t3ZYi3NvgTFmLFiOmcrvIC8eYuJInmK/MKYk/Bnd0m8MwdoUV4PDZmRohcirAWddBl1SdyCos8230lLsPggodmZNKvszI61sSx1L8i4Pelc1plWjwLs4yGdHFUVPcRUBMBTMGGjZgmRrWXn7nHqn2p1Gmozx/bz5fYnvS8udSNIKPNTPocIxcYpU++gkHFqz+gyzqOxI7qXsSJ13p0cXx/7ReDw6w3ParhpwKB4BWNlJEIMNhRDEX2ZpuJoU0kl2G4szJoH2wL8S99Jk6RJXFi9pYOhEuOQCzTF1lRqx1I+h+sF5B8IjOJkMAwsM/Ul65+SpA5xkq1pJ6FCjljvTIIqdazLFVRHML+vU0Elu1k96bPIfa4B+YetWFep6AT5JBIX2dB08oCtZ9KP97um9ESP6gboY5A6pS40bPYruAG9/AJcnXwYFqPkqCUrgBLqn/+2scrwjAvQoWMUFmtgjtPQZWWYuabetCkd1qVnfvlFPvk1HthYGb8RaWQwZV7gQHm/wOMpHGLfiscaNPaxPlEngthiXgQi3gleTpPRRNZ9WAfBZCH3JBJuS4I3ZPMOm+CL6xITMvwIKlEDzSKH4ssJA+1CK+hwT6hdM6OSXbEU1BYZtntDLnuG9K8nGSttX6soAE7eu8AUj0gS67F5ZVEWZAvLNGURljJE9hE4c1buJig0JoDNJyjNsxJXF1+C9shKjp4nXVAK2RERTnHbr/wBezNT4YJ4AAA=='
source = gzip.decompress(base64.b64decode(payload)).decode("utf-8")

runtime_path = Path("/content/sparsewalker_combined_systems_runtime_v2.py")
runtime_path.write_text(source)

spec = importlib.util.spec_from_file_location("systems_bench", runtime_path)
systems_bench = importlib.util.module_from_spec(spec)
sys.modules["systems_bench"] = systems_bench
spec.loader.exec_module(systems_bench)

print("Loaded:", runtime_path)
print("Streaming histories:", systems_bench.HISTORIES)
print("Paper-shape lengths:", systems_bench.PAPER_LENGTHS)

## 2. HSTU cached ↔ Triton parity sanity

In [ ]:
systems_bench.check_hstu_long_context_parity()
print("HSTU parity check complete.")

## 3. Cached streaming: SASRec vs HSTU vs Sparse Walker

In [ ]:
streaming_df, streaming_speedup, streaming_memory = (
    systems_bench.run_streaming_benchmark()
)

print("\nRAW STREAMING RESULTS")
display(streaming_df)

print("\nSPEEDUP TABLE")
display(streaming_speedup)

print("\nPER-USER STATE / CACHE MEMORY")
display(streaming_memory)

### How to read the streaming result

`Walker-state_speedup_vs_SASRec > 1` means Walker's state update is faster.

`Walker-terminal-d128_speedup_vs_SASRec > 1` compares the complete fused Walker
state update + sparse terminal retrieval against SASRec's state update.

Sparse Walker keeps K=8 concept IDs + masses per user, so its persistent user
state does not grow with the history-length label.

## 4. Paper-shape HSTU vs Flash/SDPA

In [ ]:
paper_df, paper_speedup = systems_bench.run_paper_shape_benchmark()

print("\nPAPER-SHAPE RAW RESULTS")
display(paper_df)

print("\nPAPER-SHAPE SPEEDUP")
display(paper_speedup)

**Keep the workloads separate:** Experiment A is cached one-event serving
(`q_len=1`). Experiment B is a full-sequence attention-core benchmark closer to
the HSTU paper's efficiency setting.

## 5. Save results to Drive

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive", force_remount=False)
OUT = Path("/content/drive/MyDrive/hstu_pure_pytorch_ml1m")
OUT.mkdir(parents=True, exist_ok=True)

streaming_df.to_csv(OUT / "sasrec_hstu_walker_a100_long_context.csv", index=False)
streaming_speedup.to_csv(
    OUT / "sasrec_hstu_walker_a100_long_context_speedup.csv", index=False
)
streaming_memory.to_csv(
    OUT / "sasrec_hstu_walker_a100_long_context_memory.csv", index=False
)
paper_df.to_csv(OUT / "hstu_paper_shape_attention_a100.csv", index=False)
paper_speedup.to_csv(
    OUT / "hstu_paper_shape_attention_a100_speedup.csv", index=False
)

print("Saved under:", OUT)